<a href="https://colab.research.google.com/github/AndrVel/simulative_python/blob/master/py_bys_case_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Кейс 2

In [1]:
import gspread
from google.oauth2.service_account import Credentials
from oauth2client.service_account import ServiceAccountCredentials
from datetime import datetime, timedelta
import os

In [2]:
!wget https://gist.github.com/Vs8th/d0bd4bdbbb58c8ae4f70a2a503e2d5fc/raw/creds.json

!wget https://gist.github.com/Vs8th/39c5deed0f5539d781f00328f7fd4fe0/raw/result.txt

--2026-06-02 03:02:51--  https://gist.github.com/Vs8th/d0bd4bdbbb58c8ae4f70a2a503e2d5fc/raw/creds.json
Resolving gist.github.com (gist.github.com)... 140.82.113.4
Connecting to gist.github.com (gist.github.com)|140.82.113.4|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://gist.githubusercontent.com/Vs8th/d0bd4bdbbb58c8ae4f70a2a503e2d5fc/raw/creds.json [following]
--2026-06-02 03:02:51--  https://gist.githubusercontent.com/Vs8th/d0bd4bdbbb58c8ae4f70a2a503e2d5fc/raw/creds.json
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2358 (2.3K) [text/plain]
Saving to: ‘creds.json’

creds.json          100%[===================>]   2.30K  --.-KB/s    in 0s      

2026-06-02 03:02:51 (26.4 MB/s) - ‘creds.json’ saved [2

In [3]:
# Указываем необходимые права доступа к таблицам
scope = ['https://www.googleapis.com/auth/spreadsheets.readonly',
         'https://www.googleapis.com/auth/drive']

# Загружаем ключи аутентификации из файла json
creds = ServiceAccountCredentials.from_json_keyfile_name('creds.json', scope)

# Авторизуемся в Google Sheets API
client = gspread.authorize(creds)

In [4]:
sheet = client.open("Installments").worksheet("Лист1")
sheet1_data = sheet.get_all_records()
sheet1_data[:2]

[{'student_id': 1, 'student_name': 'Смирнова И.И.', 'installment': 'Y'},
 {'student_id': 2, 'student_name': 'Кузнецова К.А.', 'installment': 'Y'}]

In [5]:
sheet = client.open("Installments").worksheet("Лист2")
sheet2_data = sheet.get_all_records()
# sheet2_data[:2]

In [6]:
sheet = client.open("Installments").worksheet("Лист3")
sheet3_data = sheet.get_all_records()
# sheet3_data[:2]

In [7]:
def generate_report(sheet1, sheet2, sheet3):

  # Создаю словарь max_duration, где по student_id будет проставлено количество полных просрочек
  max_duration = 183
  delays = {}
  today = datetime.strptime("01.03.2023", "%d.%m.%Y").date()

  # Заполняю словарь max_duration из листа 2 таблицы
  for item in sheet2[:]:
    last_pay_day = datetime.strptime(item['last_payment_date'], "%d.%m.%Y").date()
    day_diff = (today - last_pay_day).days
    if day_diff > max_duration:
      delays[item["student_id"]] = day_diff // max_duration

  # Прохожу по листу 3 и для каждой записи считаю сумму неоплаченного долга.
  debts = {}
  for item in sheet3:
    if item["student_id"] in delays.keys():
      debts[item["student_id"]] = item["one-time_payment"] * delays[item["student_id"]]

  # Прохожу по листу 1 и для каждого student_id, который есть в debts выбираю имя, формирую файл

  path = "student_debt_report.txt"
  mode = "w" if not os.path.exists(path) else "a"
  with open(path, mode, encoding="utf-8") as fin_file:
    for item in sheet1:
      if item['student_id'] in debts.keys():
        fin_file.write(f"Студент {item["student_name"]} - долг {debts[item['student_id']]} рублей\n")


In [8]:
generate_report(sheet1_data, sheet2_data, sheet3_data)